In [59]:
import os

import torch
import torch.optim as optim
from torch.nn import CrossEntropyLoss
from torch.nn import functional as F
from torch.optim import Adam, AdamW

os.environ["WANDB_API_KEY"] = "KEY"
os.environ["WANDB_MODE"] = 'offline'
from itertools import combinations

from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
import tqdm
from sklearn.metrics import confusion_matrix
from torch.utils.data import DataLoader, Dataset
import random
import csv
from torch import Tensor
import itertools
import math
import re
import numpy as np
import argparse
import pickle
import seaborn as sns
import neo
from quantities import ms
from elephant.statistics import instantaneous_rate
from elephant.kernels import GaussianKernel

from PIL import Image
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from sklearn.model_selection import train_test_split
import pandas as pd
from torch.utils.data import random_split

In [60]:
cluster_inf = pd.read_csv("/media/ubuntu/sda/duan/raw_data/251205/cluster_info.tsv", sep = '\t')
spike_clusters = np.load("/media/ubuntu/sda/duan/raw_data/251205/spike_clusters.npy")
spike_times = np.load("/media/ubuntu/sda/duan/raw_data/251205/spike_times.npy")

In [71]:
cluster_inf = cluster_inf[cluster_inf['KSLabel'] == 'good']
spike_inf = pd.DataFrame([spike_clusters, spike_times])
spike_inf = spike_inf.T
spike_inf.columns = ['cluster', 'time']
spike_inf = spike_inf[spike_inf['cluster'].isin(cluster_inf['cluster_id'].values)]

valid_cluster = spike_inf['cluster'].value_counts()
valid_cluster = valid_cluster[valid_cluster > 15000].index

spike_inf = spike_inf[spike_inf['cluster'].isin(valid_cluster)]
cluster_inf = cluster_inf[cluster_inf['cluster_id'].isin(valid_cluster)]

spike_inf = spike_inf[spike_inf['time'] > 1e8]

In [72]:
with PdfPages("duration.pdf") as pdf:
    for neuron in spike_inf['cluster'].unique():
        neuron_df = spike_inf[spike_inf['cluster'] == neuron]
        plt.figure(figsize = (4, 2))
        sns.histplot(neuron_df['time'])
        plt.title(neuron)
        pdf.savefig()
        plt.close()

In [73]:
cluster_stable = [336, 33, 455, 56, 31, 331, 229, 529, 54,
                  371, 216, 522, 560, 150, 217, 118, 87, 70,
                  177, 24, 279, 243, 266, 42, 399, 342, 268, 
                  293, 278, 237, 292, 84, 171, 66, 285, 65, 
                  176, 509, 101, 277, 27, 178, 157, 41]

In [76]:
spike_inf = spike_inf[spike_inf['cluster'].isin(cluster_stable)]
cluster_inf = cluster_inf[cluster_inf['cluster_id'].isin(cluster_stable)]

In [4]:
trigger = pd.read_csv("/media/ubuntu/sda/duan/raw_data/251205/rec_params.csv")
trigger = trigger[trigger['bhv_codes'] == 10]
trigger = trigger.reset_index()
stimuli_pattern = pd.read_csv('/media/ubuntu/sda/visual_stimuli_pattern/things/visual_stimuli_sequence.csv').iloc[:11427, :]
trigger = pd.concat((trigger, stimuli_pattern), axis = 1)
trigger = trigger[trigger['trial_error'] == 0]
trigger['rec_codes_points'] = trigger['rec_codes_points'] 

image = [i.split("/")[-1].split(".")[0] for i in trigger['image_path'].values]
trigger['image_id'] = image


In [69]:
trigger = trigger[trigger['rec_codes_points'] >= 1e8]

In [77]:
with PdfPages("raster_overall.pdf") as pdf:
    for neuron in spike_inf['cluster'].unique():
        neuron_df = spike_inf[spike_inf['cluster'] == neuron]

        fig, ax = plt.subplots(figsize = (12, 7.5))

        for index, row in trigger.iterrows():
            start = row['rec_codes_points'] - 150 * 30
            end = row['rec_codes_points'] + 450 * 30
            
            filtered_spikes = neuron_df[(neuron_df['time'] >= start) & 
                                        (neuron_df['time'] <= end)]
            
            if not filtered_spikes.empty:
                ax.plot(filtered_spikes['time'] - start, [index] * len(filtered_spikes), marker='|', mew=1, markersize=3, ls='', color='k')

        ax.set_yticks([])
        #ax.axvspan(1 + (5000 - 1) * 0.5, 1 + (5000 - 1), color='gray', alpha=0.3)

        ax.set_xlabel('time (s)')
        ax.set_ylabel("")
        ax.set_title(f'Raster Plot for {neuron}')

        pdf.savefig(fig)
        plt.close()

In [78]:
cluster_visual = [118, 150, 217, 87, 177, 24, 279, 243,
                  266, 399, 268, 293, 278, 237, 292, 84, 171, 66, 
                  285, 176, 509, 101, 277, 27, 178, 101, 277, 27, 178, 157, 41]

In [79]:
spike_inf_visual = spike_inf[spike_inf['cluster'].isin(cluster_visual)]

In [111]:
def create_spike_train_dict(spike_inf, trigger_time, t_start=0, t_stop=500, output_dir=None):
    """
    创建按 train/test 和 image_id 组织的 firing rate 矩阵并保存为文件（内存优化版本）
    
    保存4个文件:
    - train_firing_rate_matrix.npy: (n_image, n_neuron, time_bin)
    - train_metadata.csv: 对应的 metadata
    - test_firing_rate_matrix.npy: (total_trials, n_neuron, time_bin) - 所有 image 的所有 trial 展平
    - test_metadata.csv: 对应的 metadata，包含每个 trial 的 image_id 和 trial_index
    """
    gk = GaussianKernel(25 * ms)
    
    # 确定 image_id 列名
    image_id_col = 'test_id' if 'test_id' in trigger_time.columns else 'image_id'
    
    # 分离 train 和 test（使用视图而不是copy）
    trigger_train = trigger_time[trigger_time['test'] == 0]
    trigger_test = trigger_time[trigger_time['test'] == 1]
    
    # 获取所有 cluster（neuron）ID，并排序以确保一致性
    cluster_ids = sorted(spike_inf['cluster'].unique())
    n_neurons = len(cluster_ids)
    
    # 计算时间 bin 数量
    sampling_period_ms = 10
    # t_stop 和 t_start 的单位是 ms，sampling_period_ms 也是 ms，所以直接相除
    n_time_bins = int((t_stop - t_start) / sampling_period_ms)
    
    # 将 spike_inf 转换为 numpy 数组以提高效率
    spike_times_arr = spike_inf['time'].values
    spike_clusters_arr = spike_inf['cluster'].values
    
    # 创建 cluster_id 到索引的映射
    cluster_to_idx = {cid: idx for idx, cid in enumerate(cluster_ids)}
    
    # 处理 train 数据：按 image_id 分组，每个 image 只有 1 个 trial
    print("Processing train data...")
    train_image_list = []
    train_data_list = []
    train_metadata_rows = []
    
    # 获取所有唯一的 image_id 并排序以确保顺序一致
    train_image_ids = sorted(trigger_train[image_id_col].unique())
    for image_id in train_image_ids:
        group = trigger_train[trigger_train[image_id_col] == image_id]
        image_name = str(image_id)
        if len(train_image_list) % 1000 == 0:
            print(f"  Processing train image {image_name}: {len(group)} trials (progress: {len(train_image_list)}/{len(train_image_ids)})")
        
        # train 数据每个 image 只有 1 个 trial，直接使用
        trial_row = group.iloc[0]
        rec_codes_point = int(trial_row['rec_codes_points'])
        
        # 定义时间窗口（使用整数索引）
        start_time = rec_codes_point
        end_time = rec_codes_point + int(t_stop * 30)
        
        # 直接使用 numpy 数组索引，避免创建 DataFrame 副本
        mask = (spike_times_arr >= start_time) & (spike_times_arr < end_time)
        trial_spike_times = spike_times_arr[mask]
        trial_spike_clusters = spike_clusters_arr[mask]
        relative_times = (trial_spike_times - rec_codes_point) / 30.0  # 转换为 ms
        
        # 为每个 cluster 创建 spike train
        trial_neuron_rates = np.zeros((n_neurons, n_time_bins), dtype=np.float32)
        
        for cluster_id in cluster_ids:
            cluster_mask = trial_spike_clusters == cluster_id
            cluster_spikes_ms = relative_times[cluster_mask]
            
            valid_spikes = cluster_spikes_ms[(cluster_spikes_ms >= t_start) & (cluster_spikes_ms <= t_stop)]
            
            if len(valid_spikes) > 0:
                spike_train = neo.SpikeTrain(
                    valid_spikes.astype(int) * ms,
                    t_stop=t_stop * ms,
                    t_start=t_start * ms
                )
                
                inst_rate = instantaneous_rate(spike_train, kernel=gk, sampling_period=sampling_period_ms*ms).magnitude.flatten()
                
                # 确保长度一致并转换为 float32
                if len(inst_rate) != n_time_bins:
                    if len(inst_rate) > n_time_bins:
                        inst_rate = inst_rate[:n_time_bins]
                    else:
                        inst_rate = np.pad(inst_rate, (0, n_time_bins - len(inst_rate)))
                
                trial_neuron_rates[cluster_to_idx[cluster_id], :] = inst_rate.astype(np.float32)
        
        train_data_list.append(trial_neuron_rates)
        train_image_list.append(image_name)
        
        # 保存 metadata
        train_metadata_rows.append({
            'image_id': image_name,
            'image_path': trial_row.get('image_path', ''),
            'rec_codes_points': rec_codes_point
        })
        
        # 释放内存
        del trial_neuron_rates, trial_spike_times, trial_spike_clusters, relative_times
    
    # 堆叠所有 train images，得到 (n_image, n_neuron, time_bin)
    train_firing_rate_matrix = np.stack(train_data_list, axis=0).astype(np.float32)  # (n_image, n_neuron, time_bin)
    train_metadata = pd.DataFrame(train_metadata_rows)
    
    # 释放 train_data_list
    del train_data_list
    
    print(f"Train data shape: {train_firing_rate_matrix.shape}, dtype: {train_firing_rate_matrix.dtype}")
    
    # 处理 test 数据：按 image_id 分组，保留所有 trial
    print("Processing test data...")
    test_image_list = []
    test_data_list = []
    test_metadata_rows = []
    test_n_trials_list = []  # 记录每个 image 的实际 trial 数量
    
    # 获取所有唯一的 image_id 并排序以确保顺序一致
    test_image_ids = sorted(trigger_test[image_id_col].unique())
    for img_idx, image_id in enumerate(test_image_ids):
        group = trigger_test[trigger_test[image_id_col] == image_id]
        image_name = str(image_id)
        n_trials = len(group)
        
        if img_idx % 10 == 0:
            print(f"  Processing test image {image_name}: {n_trials} trials (progress: {img_idx}/{len(test_image_ids)})")
        
        # 预分配数组：直接创建 (n_neuron, time_bin, n_trial) 数组
        image_data = np.zeros((n_neurons, n_time_bins, n_trials), dtype=np.float32)
        
        for trial_idx, (_, trial_row) in enumerate(group.iterrows()):
            rec_codes_point = int(trial_row['rec_codes_points'])
            
            # 定义时间窗口
            start_time = rec_codes_point
            end_time = rec_codes_point + int(t_stop * 30)
            
            # 直接使用 numpy 数组索引
            mask = (spike_times_arr >= start_time) & (spike_times_arr < end_time)
            trial_spike_times = spike_times_arr[mask]
            trial_spike_clusters = spike_clusters_arr[mask]
            relative_times = (trial_spike_times - rec_codes_point) / 30.0
            
            # 为每个 cluster 创建 spike train
            for cluster_id in cluster_ids:
                cluster_mask = trial_spike_clusters == cluster_id
                cluster_spikes_ms = relative_times[cluster_mask]
                
                valid_spikes = cluster_spikes_ms[(cluster_spikes_ms >= t_start) & (cluster_spikes_ms <= t_stop)]
                
                if len(valid_spikes) > 0:
                    spike_train = neo.SpikeTrain(
                        valid_spikes.astype(int) * ms,
                        t_stop=t_stop * ms,
                        t_start=t_start * ms
                    )
                    
                    inst_rate = instantaneous_rate(spike_train, kernel=gk, sampling_period=sampling_period_ms*ms).magnitude.flatten()
                    
                    # 确保长度一致并转换为 float32
                    if len(inst_rate) != n_time_bins:
                        if len(inst_rate) > n_time_bins:
                            inst_rate = inst_rate[:n_time_bins]
                        else:
                            inst_rate = np.pad(inst_rate, (0, n_time_bins - len(inst_rate)))
                    
                    image_data[cluster_to_idx[cluster_id], :, trial_idx] = inst_rate.astype(np.float32)
            
            # 保存 metadata
            test_metadata_rows.append({
                'image_id': image_name,
                'image_path': trial_row.get('image_path', ''),
                'rec_codes_points': rec_codes_point,
                'trial_index': trial_idx
            })
            
            # 释放中间变量
            del trial_spike_times, trial_spike_clusters, relative_times
        
        # 只保留有效的 trial（前 n_trials 个），然后转置为 (n_trials, n_neuron, time_bin)
        image_data_valid = image_data[:, :, :n_trials]  # (n_neuron, time_bin, n_trials)
        # 转置为 (n_trials, n_neuron, time_bin)
        image_data_valid = np.transpose(image_data_valid, (2, 0, 1))  # (n_trials, n_neuron, time_bin)
        test_data_list.append(image_data_valid)
        test_image_list.append(image_name)
        
        # 释放内存
        del image_data, image_data_valid
    
    # 将所有 test images 的 trial 展平，得到 (total_trials, n_neuron, time_bin)
    # 使用 concatenate 而不是 stack，因为每个 image 的 trial 数量可能不同
    test_firing_rate_matrix = np.concatenate(test_data_list, axis=0).astype(np.float32)  # (total_trials, n_neuron, time_bin)
    test_metadata = pd.DataFrame(test_metadata_rows)
    # 为每个 image 添加 n_trials 信息
    image_n_trials_dict = dict(zip(test_image_list, test_n_trials_list))
    test_metadata['n_trials'] = test_metadata['image_id'].map(image_n_trials_dict)
    
    # 释放 test_data_list
    del test_data_list
    
    print(f"Test data shape: {test_firing_rate_matrix.shape}, dtype: {test_firing_rate_matrix.dtype}")
    
    # 保存文件
    if output_dir is not None:
        os.makedirs(output_dir, exist_ok=True)
        
        # 保存矩阵
        train_matrix_path = os.path.join(output_dir, 'train_firing_rate_matrix.npy')
        test_matrix_path = os.path.join(output_dir, 'test_firing_rate_matrix.npy')
        np.save(train_matrix_path, train_firing_rate_matrix)
        np.save(test_matrix_path, test_firing_rate_matrix)
        print(f"Saved train matrix to: {train_matrix_path}")
        print(f"Saved test matrix to: {test_matrix_path}")
        
        # 保存 metadata
        train_meta_path = os.path.join(output_dir, 'train_metadata.csv')
        test_meta_path = os.path.join(output_dir, 'test_metadata.csv')
        train_metadata.to_csv(train_meta_path, index=False)
        test_metadata.to_csv(test_meta_path, index=False)
        print(f"Saved train metadata to: {train_meta_path}")
        print(f"Saved test metadata to: {test_meta_path}")
    
    return {
        'train_firing_rate_matrix': train_firing_rate_matrix,
        'train_metadata': train_metadata,
        'test_firing_rate_matrix': test_firing_rate_matrix,
        'test_metadata': test_metadata
    }


In [112]:
# 创建 firing rate 矩阵并保存为文件
output_dir = '/media/ubuntu/sda/duan/result/251205_30k'  # 修改为你想要的输出目录
result = create_spike_train_dict(spike_inf_visual, trigger, t_start=0, t_stop=300, output_dir=output_dir)

# 结果包含4个部分：
# - result['train_firing_rate_matrix']: (n_image, n_neuron, time_bin)
# - result['train_metadata']: DataFrame with image_id, image_path, rec_codes_points
# - result['test_firing_rate_matrix']: (total_trials, n_neuron, time_bin) - 所有 image 的所有 trial 展平
# - result['test_metadata']: DataFrame with image_id, image_path, rec_codes_points, trial_index, n_trials


Processing train data...
  Processing train image aardvark_10s: 1 trials (progress: 0/4763)
  Processing train image coffee_table_01b: 1 trials (progress: 1000/4763)
  Processing train image hedge_13s: 1 trials (progress: 2000/4763)
  Processing train image phone_booth_01b: 1 trials (progress: 3000/4763)
  Processing train image strap_02s: 1 trials (progress: 4000/4763)
Train data shape: (4763, 27, 30), dtype: float32
Processing test data...
  Processing test image test_01: 13 trials (progress: 0/100)
  Processing test image test_100: 14 trials (progress: 10/100)
  Processing test image test_20: 19 trials (progress: 20/100)
  Processing test image test_30: 19 trials (progress: 30/100)
  Processing test image test_40: 20 trials (progress: 40/100)
  Processing test image test_50: 12 trials (progress: 50/100)
  Processing test image test_60: 17 trials (progress: 60/100)
  Processing test image test_70: 16 trials (progress: 70/100)
  Processing test image test_80: 13 trials (progress: 80/1

In [169]:
test_firing_rate_matrix = result['test_firing_rate_matrix']
test_metadata = result['test_metadata']
train_firing_rate_matrix = result['train_firing_rate_matrix']
train_metadata = result['train_metadata']

In [153]:
# 计算每个neuron在每个test_image上的trial间correlation
# 对于每个neuron的每个test_image：计算1个trial和其他trial均值的correlation，然后求平均
# 结果：correlation_matrix (n_neuron, n_image)

from scipy.stats import pearsonr

# 获取所有唯一的image_id并排序
unique_image_ids = sorted(test_metadata['image_id'].unique())
n_images = len(unique_image_ids)
n_neurons = test_firing_rate_matrix.shape[1]

# 初始化结果矩阵
correlation_matrix = np.zeros((n_neurons, n_images), dtype=np.float32)

print(f"计算 {n_neurons} 个 neuron 在 {n_images} 个 test image 上的 trial 间 correlation...")
print(f"test_firing_rate_matrix shape: {test_firing_rate_matrix.shape}")

for img_idx, image_id in enumerate(unique_image_ids):
    # 获取该image的所有trial索引
    image_mask = test_metadata['image_id'] == image_id
    image_trial_indices = np.where(image_mask)[0]
    n_trials = len(image_trial_indices)
    
    if img_idx % 10 == 0:
        print(f"  处理 image {image_id}: {n_trials} trials (progress: {img_idx}/{n_images})")
    
    # 提取该image的所有trial数据: (n_trials, n_neuron, time_bin)
    image_trials = test_firing_rate_matrix[image_trial_indices, :, :]  # (n_trials, n_neuron, time_bin)
    
    # 对于每个neuron
    for neuron_idx in range(n_neurons):
        # 提取该neuron在所有trial的firing rate: (n_trials, time_bin)
        neuron_trials = image_trials[:, neuron_idx, :]  # (n_trials, time_bin)
        
        # 计算所有trial的平均值: (time_bin,)
        trial_mean = np.mean(neuron_trials, axis=0)  # (time_bin,)
        
        # 对于每个trial，计算与平均值的correlation
        correlations = []
        for trial_idx in range(n_trials):
            trial_data = neuron_trials[trial_idx, :]  # (time_bin,)
            
            # 计算correlation
            if np.std(trial_data) > 0 and np.std(trial_mean) > 0:
                corr, _ = pearsonr(trial_data, trial_mean)
                correlations.append(corr)
            else:
                correlations.append(0.0)  # 如果方差为0，correlation设为0
        
        # 对所有trial的correlation求平均
        correlation_matrix[neuron_idx, img_idx] = np.mean(correlations)

print(f"\nCorrelation matrix shape: {correlation_matrix.shape}")
print(f"Correlation range: [{correlation_matrix.min():.4f}, {correlation_matrix.max():.4f}]")
print(f"Mean correlation: {correlation_matrix.mean():.4f}")

# # 保存结果
# correlation_output_path = '/media/ubuntu/sda/duan/result/251205_30k/neuron_image_correlation_matrix.npy'
# np.save(correlation_output_path, correlation_matrix)
# print(f"Correlation matrix saved to: {correlation_output_path}")

# # 创建对应的metadata（image_id列表）
# image_id_list = pd.DataFrame({
#     'image_id': unique_image_ids,
#     'image_index': range(n_images)
# })
# image_id_list_path = '/media/ubuntu/sda/duan/result/251205_30k/test_image_ids.csv'
# image_id_list.to_csv(image_id_list_path, index=False)
# print(f"Image ID list saved to: {image_id_list_path}")


计算 27 个 neuron 在 100 个 test image 上的 trial 间 correlation...
test_firing_rate_matrix shape: (1538, 27, 30)
  处理 image test_01: 13 trials (progress: 0/100)
  处理 image test_100: 14 trials (progress: 10/100)
  处理 image test_20: 19 trials (progress: 20/100)
  处理 image test_30: 19 trials (progress: 30/100)
  处理 image test_40: 20 trials (progress: 40/100)
  处理 image test_50: 12 trials (progress: 50/100)
  处理 image test_60: 17 trials (progress: 60/100)
  处理 image test_70: 16 trials (progress: 70/100)
  处理 image test_80: 13 trials (progress: 80/100)
  处理 image test_90: 15 trials (progress: 90/100)

Correlation matrix shape: (27, 100)
Correlation range: [-0.0166, 0.7885]
Mean correlation: 0.2380


In [170]:
test_firing_rate_matrix = test_firing_rate_matrix[:, np.where(correlation_matrix.mean(axis = 1) > 0.2)[0], :]

In [171]:
test_matrix_path = os.path.join(output_dir, 'test_firing_rate_matrix_oracle.npy')
np.save(test_matrix_path, test_firing_rate_matrix)

In [172]:
test_firing_rate_matrix.shape

(1538, 16, 30)